# nano Beatrix — the four-digit address code at 48 records a row with a higher learning-rate floor at 24,000 steps, lane 3 of 3

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_a48sa_lane3.ipynb)

**The question**: In the address-code stages each model writes an index of the records it holds, and that index forms by a sudden jump at a step that differs by seed. With the address as four digits of four at 48 records a row, the jump had not come by 24,000 steps on four seeds of six: two stopped at the fourth digit, two earlier. Cooling the learning rate to half its peak instead of a tenth formed the index within the same 24,000 steps on most of the stalled seeds of the same code at 96 records a row and of the plain-rows recipe, on the same rows at the same steps. These notebooks train the 48-record recipe again for the stored 24,000 steps under that schedule, on all six stored seeds and therefore the same rows at the same steps: does the index form here too?

| what | setting | the name carries |
|---|---|---|
| the schedule | after the same 100 warm-up steps, the cosine from 1e-3 ends at half of it (`--lr_floor=0.5`) instead of a tenth | `_lf0.5` |
| the recipe | the plain-rows recipe with the address as four digits of four, 48 records a row (`--grid_code=4x4 --grid_comps=48`) | `_gc48_ac44` |
| the seeds | 1, 2, 3, 5 (the index had not formed at 24,000 steps), then 4 and 6 (formed by 18,000 and 4,000) | `_s<seed>` |

Everything else is the stored recipe verbatim; the name carries `_a48sa`. The stored 24,000-step models of the same seeds are on [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments) and are not trained again: each new model is set beside its stored one.

**The lanes**: three notebooks, one per G4 runtime, identical but for the lane's name. Each walks the six in the order seeds 1, 2, 3, 5, 4, 6; it claims a model before training it, by a small file under `bed/preflight/a48sa_claims/` written only if the repo has not moved since the lane's read of it, so no two lanes train one model; one runtime alone trains all six in turn. A model whose run is already whole on the Hub (a stopped lane's) is fetched and only its look is read.

**The cells**: C1 pins the trainer and the arm library and holds the decision block and the lane's name; C2 builds the data, checks the stored models on the Hub, runs 200 steps of the first part on this card for the price line (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 is the lane: it claims, trains and reads each model's look as it lands; C4 renders the tables (every lane's models, and each model beside its stored 24,000-step model of the same seed). Every finished run and record ships to [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments) (`bed/reports/`, `bed/weights/<run>/` with the phases in `key_rot.safetensors`, `bed/reports/look/`).

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on: a lane claims before it trains, so a tokenless lane trains nothing. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` starts no new model; a restarted session takes back its own unfinished claim. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/bed_optim.py` - `lr_schedule` - `bed_train.py`, `names.py`; `tests/test_lr_schedule.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.100" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 100), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, the looks
REPO = 'AbstractPhil/geolip-rotary-experiments'      # every finished run, record and claim ships here (the rotary line's repo); the comparison is stored here too
LANE = 'lane3'                                        # this lane's name: its claims carry it (three G4 notebooks: lane1, lane2, lane3)
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000,                     # the formation trace every 2,000 steps (the pointer's masses read at each point)
                 hard_read=1)                          # the index on 64 key-hard rows at the end
WIDE = dict(rotary_content=4)                          # ALHENA-W: 4 position / 4 content / 8 address pairs
VALUE = dict(value_addr=1)                             # ALHENA-V: the value-path tables, born zero (the name carries _va1)
POINTER = dict(ptr_lam=0.1, ptr_block=4, ptr_head=0)  # ALHENA-P: block 4 of 6, head 0, lambda .1 (the pointer screen's rule)
HARD = dict(grid_hard=0.5)                             # ALHENA-PK: the last 2 of the batch's 4 grid rows key-hard, from birth (the name carries _kh50)
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
ROT = dict(key_rot=1)                                  # the key in the told head's rotation
SIXTEEN = dict(key_pairs=16)                           # all sixteen of the told head's pairs bound at the coded end-markers (the name carries _krt16)
RECORDS = dict(key_norm=2, key_rho=2.18)               # the key's bound pairs held at length 2.18 at the records' end-markers only (the name carries _krt16r218)
PLAIN = dict(plant=0.75, plain_doc=1)                  # three non-grid rows in four framed, the fourth a plain document opened by DOC (the name carries _p75d)
CODE = dict(grid_code='4x4')                           # the address as four digits of four (256 addresses) on the same eight address pairs (the name carries _ac44)
STEPS = 24000                                          # the stored length (the name carries _L24000)
FLOOR = ('_lf0.5', dict(lr_floor=0.5))                 # the cosine from 1e-3 ends at .5 x 1e-3 instead of .1
SERIES = 'a48sa'
_T = lambda sp: TAIL(STEPS).replace('_askany_', '_askany_p75d_').replace(f'_L{STEPS}_', f'_L{STEPS}{sp}_', 1)
TAG_A48 = lambda s, sp, series: f'alhena-both_s{s}{_T(sp)}_cp4_gc48_ac44_kh50_va1_pt10b4h0_krt16r218_ahdigit1_{series}'
A48 = lambda s: (f'A48 FLOOR s{s}', TAG_A48(s, FLOOR[0], SERIES), s, dict(BED, **GRID_ROWS, **HARD, **WIDE, **VALUE, **POINTER, **ROT, **SIXTEEN, **RECORDS, **PLAIN, **CODE, grid_comps=48, steps=STEPS, **FLOOR[1], arms='alhena-both', seeds=str(s), series=SERIES))
PARTS = [A48(s) for s in (1, 2, 3, 5, 4, 6)]           # the plan's order: the four seeds whose index had not formed at 24,000 steps, then the two that had
STORED = {('A48', s): TAG_A48(s, '', 'code44') for s in range(1, 7)}      # the stored 24,000-step models of the same seeds on REPO (never trained again)
LOOK_HEAD = (4, 0)                                     # block 4 (numbered from 1), head 0: the head the term trains
READ_BATCHES, CALIB_BATCHES, HARD_BATCHES = 64, 32, 16 # 512 read rows, 256 calibration rows, 128 key-hard rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of the first part on this card for the price line; its record stays local
PLAN_MIN_24K = 19.0                                    # the planned minutes of one 24,000-step run at 48 records a row on this card (the code's measured pace)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block:', [p[0] for p in PARTS], '| records a row', [p[3]['grid_comps'] for p in PARTS], '| bound pairs', [p[3].get('key_pairs', 8) for p in PARTS], '| the hold', [p[3].get('key_norm', 0) for p in PARTS], '| its length', [p[3].get('key_rho') for p in PARTS], '| the share framed', [p[3].get('plant') for p in PARTS], '| plain rows opened by DOC', [p[3].get('plain_doc', 0) for p in PARTS], '| key-hard rows', [p[3].get('grid_hard', 0) for p in PARTS],
      '| the address code', [p[3].get('grid_code', '8x2') for p in PARTS], '| steps', [p[3]['steps'] for p in PARTS],
      '| the floor', [p[3].get('lr_floor', 0.1) for p in PARTS], '| the flat share', [p[3].get('lr_flat', 0) for p in PARTS], '| lane', LANE,
      '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of the first part on this card (the price line), the push probe. No part trains.
import json, os, shutil, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
from geolip.nano_beatrix import rotary_scale as rs
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
# the comparison: the stored 24,000-step models of the same seeds, their weights, phases, records and looks on the Hub (never trained again here)
from huggingface_hub import HfApi as _Api
_hub = set(_Api().list_repo_files(REPO))
_tw = [t for t in STORED.values() if not all(f in _hub for f in (f'bed/weights/{t}/model.safetensors', f'bed/weights/{t}/key_rot.safetensors',
       f'bed/reports/bed_{t}.json', f'bed/reports/look/{t}_look_b4h0.json'))]
assert not _tw, ('the stored models are not all on the Hub', _tw)
print('the comparison: the four-digit address at 48 records a row, seeds 1-6 at 24,000 steps on', REPO, '(weights, phases, records, looks): 6 of 6')
_mine = [f for f in _hub if f.startswith('bed/') and '_a48sa' in f]
print('this stage on', REPO, 'before this session:', len(_mine), 'files')
todo_steps = 0
for label, tag, seed, spec in PARTS:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo_steps += (p['runs'] - p['done_before']) * spec['steps']
smoke = dict(PARTS[0][3], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2, hard_batches=1)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_24k = 24000 * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_24k <= STOP_FACTOR * PLAN_MIN_24K
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one 24,000-step run about {per_24k:.1f} min (planned "
      f"{PLAN_MIN_24K}); the {todo_steps:,} steps left about "
      f"{per_24k * todo_steps / 24000 / 60:.1f} h on one card, about a third of that on three lanes (+ a few minutes of end reads and the look read a run)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'bed/preflight/push_probe_a48sa_{LANE}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [p[0] for p in PARTS]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'nano a48sa: push probe ({LANE})')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

def ship_file(path, dst_dir, ship):
    # one record to dst_dir on the Hub: staged alone in its own folder, so the upload carries only it
    st = f'{OUT}/stage/{os.path.basename(path)[:-5]}'
    os.makedirs(st, exist_ok=True)
    shutil.copy(path, st)
    ship.submit(st, dst_dir)

In [ ]:
# C3 (T) — THE LANE: the six models in the plan's order (seeds 1, 2, 3, 5, whose index had not formed at 24,000 steps, then 4 and 6), one
# at a time, each claimed on the Hub before it trains, so one runtime trains all six and three divide them. Before each model the Hub is read at one commit: a model
# with its look record there (or in OUT/look/) is done; a model with another lane's claim younger than FRESH_S is taken; the first model
# with neither is claimed (a small file under CLAIMS, written only if the repo has not moved since that read - else the read repeats; a
# lane's own claim on an unfinished model, from before a restart, is still its). A claimed model trains resume-first (runner_done.txt in
# OUT); a model whose run already sits whole on the Hub (a stopped lane's) is fetched, never trained again, and only its look is read.
# Every finished run ships to REPO as it lands, its look read (the told head's request on 512 read rows and 128 key-hard rows) right after
# it, shipped to bed/reports/look/ (background shippers: the card never waits on the Hub). A STOP file in OUT starts no new model; the
# lane ends when no model is free. The time left is priced by the unclaimed models at this lane's measured pace.
import random
from huggingface_hub import snapshot_download
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
assert HF_TOKEN and PUSH_OK, 'a lane claims before it trains: it needs the token and the push probe read back (C2)'
shipper = globals().get('shipper') or bed_runner._Shipper(REPO, OUT, print)
for rec in bed_runner.pending_records(OUT):
    shipper.submit(rec)
lship = rs.Shipper(REPO)
CLAIMS = 'bed/preflight/a48sa_claims'
FRESH_S = 100 * 60                                     # a claim this old with no look record is a stopped lane's: the model is free again (a 24,000-step model ~20 min)

def snap():
    # the repo's head commit and its files, read at that one commit
    sha = api.repo_info(REPO).sha
    return sha, set(api.list_repo_files(REPO, revision=sha))

def look_out(tag):
    return f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'

def done(tag, files):
    return f'bed/reports/look/{rs.look_name(tag, LOOK_HEAD)}.json' in files or os.path.exists(look_out(tag))

def claims(label, files):
    # the fresh claims on a model, [(lane, utc seconds)], from the claim files' names: <label>__<lane>__<utc seconds>.json
    pre = f"{CLAIMS}/{label.replace(' ', '_')}__"
    out = []
    for f in files:
        if f.startswith(pre) and f.endswith('.json'):
            lane, utc = f[len(pre):-len('.json')].split('__')
            if time.time() - int(utc) < FRESH_S:
                out.append((lane, int(utc)))
    return out

def on_hub(tag, files):
    # a run that landed whole on the Hub: its weights, its phases, its scale.json and its run's record
    return all(f in files for f in (f'bed/weights/{tag}/model.safetensors', f'bed/weights/{tag}/key_rot.safetensors',
                                    f'bed/weights/{tag}/scale.json', f'bed/reports/bed_{tag}.json'))

def pick():
    # -> (the part claimed by this lane, the Hub's files at the claim, the unclaimed models left), or None when no model is left for it
    for attempt in range(40):
        sha, files = snap()
        for p in PARTS:                                # this lane's own claim on an unfinished model (a restarted session) is still its
            if not done(p[1], files) and any(lane == LANE for lane, _ in claims(p[0], files)):
                return p, files, sum(1 for q in PARTS if q is not p and not done(q[1], files) and not claims(q[0], files))
        free = [p for p in PARTS if not done(p[1], files) and not claims(p[0], files)]
        if not free:
            return None
        p = free[0]
        path = f"{CLAIMS}/{p[0].replace(' ', '_')}__{LANE}__{int(time.time())}.json"
        try:
            api.upload_file(path_or_fileobj=json.dumps({'lane': LANE, 'model': p[1]}).encode(), path_in_repo=path, repo_id=REPO,
                            commit_message=f'nano a48sa: {LANE} claims {p[0]}', parent_commit=sha)
            return p, files, len(free) - 1
        except Exception as e:                          # the repo moved since the read (another lane's claim or a ship) or the call failed:
            print('claim retry', attempt + 1, p[0], type(e).__name__)   # read again (a claim that landed anyway is found as this lane's)
            time.sleep(random.uniform(3, 12))
    raise RuntimeError('no claim after 40 tries')

done_here, t_all, n_here = [], time.time(), 0
while True:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new model starts')
        break
    got = pick()
    if got is None:
        print('no model is left for this lane: the lane ends')
        break
    (label, tag, seed, spec), files, left = got
    t0 = time.time()
    if not os.path.exists(f'{OUT}/bed/{tag}/model.safetensors') and on_hub(tag, files):
        print(f'=================== {label}: its run is whole on the Hub (a stopped lane\'s): fetched; only its look is read here ===================')
        snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*', f'bed/reports/bed_{tag}.json'], local_dir=f'{OUT}/w', token=HF_TOKEN)
        folder, rec_path = f'{OUT}/w/bed/weights/{tag}', f'{OUT}/w/bed/reports/bed_{tag}.json'
    else:
        print(f"=================== {label} ({spec['steps']:,} steps; this lane's model {n_here + 1}) ===================")
        res = bed_runner.run(spec, out=OUT, push=REPO, parallel=PARALLEL, shipper=shipper)
        done_here.append((label, res))
        if res['failed'] is not None or res['stopped']:
            print('the part did not finish:', label, {k: v for k, v in res.items() if k != 'ran'})
            break
        assert os.path.exists(f'{OUT}/bed/{tag}/model.safetensors'), ('the run did not leave the expected folder', tag)
        folder, rec_path = f'{OUT}/bed/{tag}', f'{OUT}/bed/bed_{tag}.json'
    out = look_out(tag)
    rec = rs.look_read(folder, out, seed, HELD, look='ptr', read_batches=READ_BATCHES, hard_batches=HARD_BATCHES, log=lambda s: None)
    assert (rec['look']['block'], rec['look']['head']) == LOOK_HEAD, (label, rec['look'])
    ship_file(out, 'bed/reports/look', lship)
    n_here += 1
    q, h = rec['read']['request'], rec['hard']['request']
    pace = (time.time() - t_all) / n_here
    _r = json.load(open(rec_path, encoding='utf-8'))
    kru = (_r['grid'].get('key_rot') or {}).get('use') or {}
    g = _r['grid']['read']
    print(f"{label} (the address code {spec.get('grid_code', '8x2')}, {spec['grid_comps']} records a row, {spec['steps']:,} steps, floor {spec.get('lr_floor', 0.1)}, flat {spec.get('lr_flat', 0)}): the answer whole at the true request "
          f"{g['true']['whole']:.4f}, at the null request {g['null_request']['whole']:.4f}; the key pairs score {kru.get('key_true', float('nan')):.2f} "
          f"toward the true record, {kru.get('key_other', float('nan')):.2f} toward the others; the look on the model's own row {q['LOOKF']['right']:.4f} "
          f"(composed row {q['LOOK']['right']:.4f}; on the key-hard rows {h['LOOKF']['right']:.4f}); the written index {q['INDEXF']['right']:.4f}; "
          f"{(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent on this lane; {left} models unclaimed, about "
          f"{pace * left / 3600:.1f} h if this lane alone trains them")
    _w = _r['heldout_bpb_by_position']['row']
    print(f"{label} text: held-out bits {_r['heldout_bpb']:.4f}; the row's windows 0-64 {_w['0-64']:.3f}, 256-1,024 {_w['256-1024']:.3f}; "
          f"the plain-collapse flag {_r['plain_collapse']['flag']}")
shipper.drain(print)
unshipped = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
print('UNSHIPPED runs:', unshipped if unshipped else 'none')
print('look records shipped' if lship.drain() else 'look records still retrying in the background')
print('this lane:', n_here, 'models read;', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done_here])

In [ ]:
# C4 — THE TABLES (the six models: this lane's, and the other lanes' read from the Hub; each model beside its stored
# 24,000-step model of the same seed): the models (bits, the frame copy, the written index both ways, the key-hard rows both ways, the answers, the told head's
# mass at steps 2,000 / 4,000 / the end read, minutes), their looks (the requests' right, the look's no-request and wrong-record shares,
# the answers through each, the key-hard rows), and the key
# rotation (the told head's scores at the read rows' index entries by pair group, the key pairs' norms). Run any time.
import glob, json
from IPython.display import Markdown, display
from huggingface_hub import hf_hub_download

def _rec(local, hub_path):
    # a record made here, or another lane's (or a stored model's) from the Hub; None when it has not landed
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))
    try:
        return json.load(open(hf_hub_download(REPO, hub_path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    except Exception:
        return None

rows = ['| run | bits | the row 0-64 / 256-1,024 (collapse) | copy 32 / 128 / 256 / 512 | index TF / FR | key-hard rows TF / FR | answers true / via index / null request | pointer mass 2k / 4k / read | min |',
        '|---|---|---|---|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    r = _rec(f'{OUT}/bed/bed_{tag}.json', f'bed/reports/bed_{tag}.json')
    if r is None:
        continue
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    hd = g.get('hard')
    hard = f"{hd['index_right']:.4f} / {hd['indexf_right']:.4f}" if hd else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f} / {g['null_request']['whole']:.4f}"
    pt_ = r['grid'].get('ptr') or {}
    tm = {t['step']: t.get('ptr_mass') for t in r['grid']['trace']}
    f_ = lambda v: '-' if v is None else f'{v:.3f}'
    ptr = f"{f_(tm.get(2000))} / {f_(tm.get(4000))} / {f_(pt_.get('mass_read'))}"
    _w = r['heldout_bpb_by_position']['row']
    rows.append(f"| {label} | {r['heldout_bpb']:.4f} | {_w['0-64']:.3f} / {_w['256-1024']:.3f} ({r['plain_collapse']['flag']}) | {copy} | {g['true']['index_right']:.4f} / {fr.get('indexf_right', float('nan')):.4f} | "
                f"{hard} | {via} | {ptr} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| model | written index (free-running) | look: composed / free-running / at the question | free-running look: none / wrong record | '
        'answers: true / written / look | look mass (decoded) / split / duplicate | key-hard rows: written / look | min |',
        '|---|---|---|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    r = _rec(f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json', f'bed/reports/look/{rs.look_name(tag, LOOK_HEAD)}.json')
    if r is None:
        continue
    q, a, lk = r['read']['request'], r['read']['answers'], r['read']['look']['LOOKF']
    h = r.get('hard', {}).get('request', {})
    hard = f"{h['INDEXF']['right']:.4f} / {h['LOOKF']['right']:.4f}" if h else '-'
    rows.append(f"| {label} | {q['INDEXF']['right']:.4f} | {q['LOOK']['right']:.4f} / {q['LOOKF']['right']:.4f} / {q['LOOKQF']['right']:.4f} | "
                f"{lk['no_request']:.4f} / {lk['wrong_record']:.4f} | {a['LAYOUT']:.4f} / {a['INDEXF']:.4f} / {a['LOOKF']:.4f} | "
                f"{lk['decoded_mass_mean']:.3f} / {lk['split']:.4f} / {lk['duplicate']:.4f} | {hard} | {r['wall_s'] / 60:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| run | index entries | the key pairs: true record / others / itself | the address pairs: true / others | key-pair norm q / k at coded end-markers (elsewhere) |',
        '|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    r = _rec(f'{OUT}/bed/bed_{tag}.json', f'bed/reports/bed_{tag}.json')
    if r is None:
        continue
    u = (r['grid'].get('key_rot') or {}).get('use') or {}
    if not u:
        continue
    rows.append(f"| {label} | {u['entries']} | {u['key_true']:.2f} / {u['key_other']:.2f} / {u['key_self']:.2f} | {u['anchor_true']:.2f} / "
                f"{u['anchor_other']:.2f} | {u['q_key_norm_coded']:.2f} / {u['k_key_norm_coded']:.2f} ({u['q_key_norm_other']:.2f} / "
                f"{u['k_key_norm_other']:.2f}) |")
display(Markdown('\n'.join(rows)))
def _pair(tag):
    # (held-out bits, the written index teacher-forced, the look at the question, the first step the written index read >= .99) of a run
    # made here or on the Hub; None where it has not landed
    r = _rec(f'{OUT}/bed/bed_{tag}.json', f'bed/reports/bed_{tag}.json')
    k = _rec(f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json', f'bed/reports/look/{rs.look_name(tag, LOOK_HEAD)}.json')
    first = None
    if r:
        first = next((t['step'] for t in r['grid']['trace'] if (t.get('index_right') or 0) >= 0.99), None)
    return (r['heldout_bpb'] if r else None, r['grid']['read']['true']['index_right'] if r else None,
            k['read']['request']['LOOKQF']['right'] if k else None, first)

f4 = lambda v: '-' if v is None else f'{v:.4f}'
fs = lambda v: 'not formed' if v is None else f'{v:,}'
rows = ['| model | bits: stored / this | bits: this - stored | written index: stored / this | formed at (>= .99): stored / this | '
        'look at the question: stored / this |', '|---|---|---|---|---|---|']
for label, tag, s, spec in PARTS:
    arm = label.split()[0]
    a, b = _pair(STORED[(arm, s)]), _pair(tag)
    d = f'{b[0] - a[0]:+.4f}' if a[0] is not None and b[0] is not None else '-'
    rows.append(f'| {label} | {f4(a[0])} / {f4(b[0])} | {d} | {f4(a[1])} / {f4(b[1])} | {fs(a[3]) if a[0] is not None else "-"} / '
                f'{fs(b[3]) if b[0] is not None else "-"} | {f4(a[2])} / {f4(b[2])} |')
display(Markdown('\n'.join(rows)))